# Rossmann Store Sales — Етап 1: Очищення та підготовка даних

Цей ноутбук реалізує **Етап 1** плану аналізу: завантаження, об'єднання, очищення та збагачення даних новими ознаками. Результат — файл `rossmann_clean.csv`, який буде використовуватись у всіх наступних етапах (EDA, A/B-тест, сегментація, дашборд).

**Вхідні файли** (мають лежати в папці `data/`):
- `train.csv`
- `store.csv`


In [6]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 120)


## 1. Завантаження даних

In [7]:
import os
print(os.getcwd())
print(os.listdir())

/Users/stetsenko/Documents/rossmann_project/notebooks
['02_heatmap_month_dow.png', '02_sales_per_customer_hist.png', '01_data_cleaning.ipynb', '02_monthly_sales_trend.png', '02_eda_network_level.ipynb', '02_sales_by_dow.png', '02_seasonal_decomposition.png', 'rossmann_eda_start.py', '01_data_cleaning_2.ipynb']


In [8]:
DATA_DIR = "../data"

train = pd.read_csv(f"{DATA_DIR}/train.csv", parse_dates=["Date"], low_memory=False)
store = pd.read_csv(f"{DATA_DIR}/store.csv")

print(f"train: {train.shape[0]:,} рядків, {train.shape[1]} колонок")
print(f"store: {store.shape[0]:,} рядків, {store.shape[1]} колонок")


train: 1,017,209 рядків, 9 колонок
store: 1,115 рядків, 10 колонок


In [9]:
train.head()

,Store,DayOfWeek,Date,Sales,Customers,Open,Promo,StateHoliday,SchoolHoliday
0,1,5,2015-07-31,5263,555,1,1,0,1
1,2,5,2015-07-31,6064,625,1,1,0,1
2,3,5,2015-07-31,8314,821,1,1,0,1
3,4,5,2015-07-31,13995,1498,1,1,0,1
4,5,5,2015-07-31,4822,559,1,1,0,1


In [10]:
store.head()

,Store,StoreType,Assortment,CompetitionDistance,CompetitionOpenSinceMonth,CompetitionOpenSinceYear,Promo2,Promo2SinceWeek,Promo2SinceYear,PromoInterval
0,1,c,a,1270.0,9.0,2008.0,0,NaN,NaN,NaN
1,2,a,a,570.0,11.0,2007.0,1,13.0,2010.0,"Jan,Apr,Jul,Oct"
2,3,a,a,14130.0,12.0,2006.0,1,14.0,2011.0,"Jan,Apr,Jul,Oct"
3,4,c,c,620.0,9.0,2009.0,0,NaN,NaN,NaN
4,5,a,a,29910.0,4.0,2015.0,0,NaN,NaN,NaN


## 2. Об'єднання таблиць

Приєднуємо характеристики магазину (`store`) до кожного запису продажів (`train`) через ключ `Store`.

In [11]:
df = train.merge(store, on="Store", how="left")
print(f"Після об'єднання: {df.shape[0]:,} рядків, {df.shape[1]} колонок")
df.head()


Після об'єднання: 1,017,209 рядків, 18 колонок


,Store,DayOfWeek,Date,Sales,Customers,Open,Promo,StateHoliday,SchoolHoliday,StoreType,Assortment,CompetitionDistance,CompetitionOpenSinceMonth,CompetitionOpenSinceYear,Promo2,Promo2SinceWeek,Promo2SinceYear,PromoInterval
0,1,5,2015-07-31,5263,555,1,1,0,1,c,a,1270.0,9.0,2008.0,0,NaN,NaN,NaN
1,2,5,2015-07-31,6064,625,1,1,0,1,a,a,570.0,11.0,2007.0,1,13.0,2010.0,"Jan,Apr,Jul,Oct"
2,3,5,2015-07-31,8314,821,1,1,0,1,a,a,14130.0,12.0,2006.0,1,14.0,2011.0,"Jan,Apr,Jul,Oct"
3,4,5,2015-07-31,13995,1498,1,1,0,1,c,c,620.0,9.0,2009.0,0,NaN,NaN,NaN
4,5,5,2015-07-31,4822,559,1,1,0,1,a,a,29910.0,4.0,2015.0,0,NaN,NaN,NaN


## 3. Первинна діагностика якості даних

In [12]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1017209 entries, 0 to 1017208
Data columns (total 18 columns):
 #   Column                     Non-Null Count    Dtype         
---  ------                     --------------    -----         
 0   Store                      1017209 non-null  int64         
 1   DayOfWeek                  1017209 non-null  int64         
 2   Date                       1017209 non-null  datetime64[ns]
 3   Sales                      1017209 non-null  int64         
 4   Customers                  1017209 non-null  int64         
 5   Open                       1017209 non-null  int64         
 6   Promo                      1017209 non-null  int64         
 7   StateHoliday               1017209 non-null  object        
 8   SchoolHoliday              1017209 non-null  int64         
 9   StoreType                  1017209 non-null  object        
 10  Assortment                 1017209 non-null  object        
 11  CompetitionDistance        1014567 no

In [13]:
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_report = pd.DataFrame({"missing_count": missing, "missing_pct": missing_pct})
missing_report[missing_report["missing_count"] > 0].sort_values("missing_pct", ascending=False)


,missing_count,missing_pct
Promo2SinceWeek,508031,49.94
Promo2SinceYear,508031,49.94
PromoInterval,508031,49.94
CompetitionOpenSinceMonth,323348,31.79
CompetitionOpenSinceYear,323348,31.79
CompetitionDistance,2642,0.26


In [14]:
# Скільки днів магазини були закриті
closed_days = (df["Open"] == 0).sum()
open_days = (df["Open"] == 1).sum()
print(f"Закритих днів: {closed_days:,} ({closed_days/len(df)*100:.1f}%)")
print(f"Відкритих днів: {open_days:,} ({open_days/len(df)*100:.1f}%)")


Закритих днів: 172,817 (17.0%)
Відкритих днів: 844,392 (83.0%)


In [ ]:
# Перевірка: чи є продажі в дні, коли магазин нібито закритий (аномалія)
anomaly = df[(df["Open"] == 0) & (df["Sales"] > 0)]
print(f"anormal sales (Open=0, але Sales>0): {len(anomaly)}")

# Перевірка: чи є нульові продажі у відкритих магазинах (теж варто подивитись)
zero_sales_open = df[(df["Open"] == 1) & (df["Sales"] == 0)]
print(f"Відкритих магазинів з нульовими продажами: {len(zero_sales_open)}")


Аномальних рядків (Open=0, але Sales>0): 0
Відкритих магазинів з нульовими продажами: 54


## 4. Обробка пропусків

- `CompetitionDistance` — невелика кількість пропусків (~0.3%). Найімовірніше означає відсутність (поки що) конкурента поруч. Заповнюємо великим значенням (медіана дуже далеко не підходить — краще підставити максимум датасету або окрему позначку) — тут використовуємо підхід "нема даних → вважаємо, що конкурент дуже далеко".
- `CompetitionOpenSinceMonth/Year` — пропуски означають, що дата відкриття конкурента невідома. Заповнюємо 0, і окремо створимо ознаку-прапорець.
- `Promo2SinceWeek/Year`, `PromoInterval` — пропуски логічні: якщо `Promo2 == 0`, магазин не бере участі в акції, тому додаткових даних немає. Заповнюємо 0 / порожнім рядком.

In [16]:
# CompetitionDistance: заповнюємо великим значенням (в 2-3 рази більше за максимум) + прапорець
max_dist = df["CompetitionDistance"].max()
df["CompetitionDistance_missing"] = df["CompetitionDistance"].isnull().astype(int)
df["CompetitionDistance"] = df["CompetitionDistance"].fillna(max_dist * 2)

# CompetitionOpenSinceMonth/Year: заповнюємо 0 + прапорець "дата невідома"
df["CompetitionOpenSince_missing"] = df["CompetitionOpenSinceMonth"].isnull().astype(int)
df["CompetitionOpenSinceMonth"] = df["CompetitionOpenSinceMonth"].fillna(0).astype(int)
df["CompetitionOpenSinceYear"] = df["CompetitionOpenSinceYear"].fillna(0).astype(int)

# --- Виправлення відомого артефакту в даних: CompetitionOpenSinceYear == 1900 ---
# Це ізольоване, явно неправдоподібне значення (округла "заглушка", а не реальна дата) —
# найближчий наступний рік у даних 1961, а основний масив починається з 1990.
# Позначаємо такі рядки як "дата невідома" замість того, щоб рахувати конкурента 115-річним.
implausible_year_mask = df["CompetitionOpenSinceYear"] == 1900
n_implausible = implausible_year_mask.sum()
print(f"Рядків з неправдоподібним CompetitionOpenSinceYear=1900: {n_implausible}")

df.loc[implausible_year_mask, "CompetitionOpenSince_missing"] = 1
df.loc[implausible_year_mask, "CompetitionOpenSinceMonth"] = 0
df.loc[implausible_year_mask, "CompetitionOpenSinceYear"] = 0

# Promo2SinceWeek/Year: логічні пропуски, якщо Promo2 == 0
df["Promo2SinceWeek"] = df["Promo2SinceWeek"].fillna(0).astype(int)
df["Promo2SinceYear"] = df["Promo2SinceYear"].fillna(0).astype(int)
df["PromoInterval"] = df["PromoInterval"].fillna("")

print("Пропуски після обробки:")
print(df.isnull().sum()[df.isnull().sum() > 0])


Рядків з неправдоподібним CompetitionOpenSinceYear=1900: 758
Пропуски після обробки:
Series([], dtype: int64)


## 5. Створення похідних ознак (feature engineering)

Додаємо ознаки, які знадобляться на наступних етапах аналізу (сезонність, ефект конкуренції, ефект промо-кампаній).

In [17]:
# --- Часові ознаки ---
df["Year"] = df["Date"].dt.year
df["Month"] = df["Date"].dt.month
df["Day"] = df["Date"].dt.day
df["WeekOfYear"] = df["Date"].dt.isocalendar().week.astype(int)


### 4.1. Перевірка правдоподібності `CompetitionOpenSinceYear`

Перед розрахунком `CompetitionOpenMonths` перевіряємо, чи немає неправдоподібних років відкриття конкурента (наприклад, явних заглушок типу 1900, які трапляються у вихідних даних Rossmann і не є реальною датою).

In [18]:
# Перевірка діапазону років (серед магазинів, де дата відома)
known_years = df.loc[df["CompetitionOpenSince_missing"] == 0, "CompetitionOpenSinceYear"]
print("Розподіл років (найменші значення):")
print(known_years.sort_values().unique()[:10])

# Рік 1900 явно є заглушкою/помилкою введення (а не реальною датою відкриття конкурента) —
# на відміну від, наприклад, 1961 чи 1990, які цілком правдоподібні для старих будівель у Європі.
IMPLAUSIBLE_YEAR_THRESHOLD = 1950
implausible = df[(df["CompetitionOpenSince_missing"] == 0) &
                  (df["CompetitionOpenSinceYear"] < IMPLAUSIBLE_YEAR_THRESHOLD)]
print(f"\nМагазинів з роком відкриття конкурента < {IMPLAUSIBLE_YEAR_THRESHOLD}: {implausible['Store'].nunique()}")
print(implausible[["Store", "CompetitionOpenSinceMonth", "CompetitionOpenSinceYear"]].drop_duplicates())

# Позначаємо ці випадки як "дата невідома", а не як реальну (хибну) дату
mask_implausible = (df["CompetitionOpenSinceYear"] < IMPLAUSIBLE_YEAR_THRESHOLD) & (df["CompetitionOpenSince_missing"] == 0)
df.loc[mask_implausible, "CompetitionOpenSince_missing"] = 1
df.loc[mask_implausible, "CompetitionOpenSinceMonth"] = 0
df.loc[mask_implausible, "CompetitionOpenSinceYear"] = 0

print(f"\nВиправлено рядків: {mask_implausible.sum()}")

Розподіл років (найменші значення):
[1961 1990 1994 1995 1998 1999 2000 2001 2002 2003]

Магазинів з роком відкриття конкурента < 1950: 0
Empty DataFrame
Columns: [Store, CompetitionOpenSinceMonth, CompetitionOpenSinceYear]
Index: []

Виправлено рядків: 0


In [19]:
# --- Скільки місяців конкурент вже працює на момент цього запису продажів ---
def months_since_competition(row):
    if row["CompetitionOpenSince_missing"] == 1:
        return np.nan
    comp_start = pd.Timestamp(year=int(row["CompetitionOpenSinceYear"]),
                               month=int(row["CompetitionOpenSinceMonth"]), day=1)
    diff = (row["Date"].year - comp_start.year) * 12 + (row["Date"].month - comp_start.month)
    return max(diff, 0)  # від'ємні значення (конкурент ще не відкрився) прирівнюємо до 0

df["CompetitionOpenMonths"] = df.apply(months_since_competition, axis=1)


In [20]:
# Перевірка: чи не залишилось неправдоподібних значень CompetitionOpenMonths
print("Максимум CompetitionOpenMonths:", df["CompetitionOpenMonths"].max())
print("Кількість значень > 300 місяців (25+ років):", (df["CompetitionOpenMonths"] > 300).sum())
print("Негативних значень:", (df["CompetitionOpenMonths"] < 0).sum())


Максимум CompetitionOpenMonths: 645.0
Кількість значень > 300 місяців (25+ років): 1004
Негативних значень: 0


In [21]:
# --- Скільки тижнів триває довгострокова акція Promo2 на момент цього запису ---
def weeks_since_promo2(row):
    if row["Promo2"] == 0 or row["Promo2SinceYear"] == 0:
        return 0
    promo_start = pd.Timestamp.fromisocalendar(int(row["Promo2SinceYear"]), int(row["Promo2SinceWeek"]), 1)
    diff_weeks = (row["Date"] - promo_start).days // 7
    return max(diff_weeks, 0)

df["Promo2Weeks"] = df.apply(weeks_since_promo2, axis=1)


In [22]:
# --- Чи потрапляє поточний місяць у список місяців акції PromoInterval ---
month_map = {1: "Jan", 2: "Feb", 3: "Mar", 4: "Apr", 5: "May", 6: "Jun",
             7: "Jul", 8: "Aug", 9: "Sep", 10: "Oct", 11: "Nov", 12: "Dec"}

def is_promo_month(row):
    if row["Promo2"] == 0 or row["PromoInterval"] == "":
        return 0
    months = row["PromoInterval"].split(",")
    return int(month_map[row["Month"]] in months)

df["IsPromoMonth"] = df.apply(is_promo_month, axis=1)


In [23]:
# --- Середній чек (продажі на одного клієнта) ---
# Обережно з діленням на нуль у закритих магазинах
df["SalesPerCustomer"] = np.where(df["Customers"] > 0, df["Sales"] / df["Customers"], 0)


In [24]:
df[["Date", "Store", "Sales", "Customers", "SalesPerCustomer",
    "CompetitionOpenMonths", "Promo2Weeks", "IsPromoMonth"]].head(10)


,Date,Store,Sales,Customers,SalesPerCustomer,CompetitionOpenMonths,Promo2Weeks,IsPromoMonth
0,2015-07-31,1,5263,555,9.482883,82.0,0,0
1,2015-07-31,2,6064,625,9.702400,92.0,278,1
2,2015-07-31,3,8314,821,10.126675,103.0,225,1
3,2015-07-31,4,13995,1498,9.342457,70.0,0,0
4,2015-07-31,5,4822,559,8.626118,3.0,0,0
5,2015-07-31,6,5651,589,9.594228,19.0,0,0
6,2015-07-31,7,15344,1414,10.851485,27.0,0,0
7,2015-07-31,8,8492,833,10.194478,9.0,0,0
8,2015-07-31,9,8565,687,12.467249,179.0,0,0
9,2015-07-31,10,7185,681,10.550661,70.0,0,0


## 6. Пошук аномалій / викидів

Перевіряємо, чи є магазини з екстремальними значеннями продажів (можуть бути помилки або дійсно унікальні магазини — вирішимо на етапі EDA, чи виключати їх).

In [25]:
open_df = df[df["Open"] == 1].copy()

sales_mean = open_df["Sales"].mean()
sales_std = open_df["Sales"].std()
open_df["Sales_zscore"] = (open_df["Sales"] - sales_mean) / sales_std

outliers = open_df[open_df["Sales_zscore"].abs() > 3]
print(f"Кількість днів-викидів (|z| > 3): {len(outliers):,} ({len(outliers)/len(open_df)*100:.2f}%)")
print(f"Кількість унікальних магазинів серед викидів: {outliers['Store'].nunique()}")


Кількість днів-викидів (|z| > 3): 13,437 (1.59%)
Кількість унікальних магазинів серед викидів: 409


## 7. Фінальна перевірка та збереження очищеного датасету

In [26]:
# Фінальна перевірка CompetitionOpenMonths на правдоподібність
print("CompetitionOpenMonths — перевірка:")
print(f"  Негативні значення: {(df['CompetitionOpenMonths'] < 0).sum()}")
print(f"  Максимум: {df['CompetitionOpenMonths'].max()}")
print(f"  NaN (дата невідома): {df['CompetitionOpenMonths'].isna().sum()}")


CompetitionOpenMonths — перевірка:
  Негативні значення: 0
  Максимум: 645.0
  NaN (дата невідома): 324106


In [27]:
print("Фінальна форма датасету:", df.shape)
print("\nТипи даних:")
print(df.dtypes)
print("\nПропуски, що залишились:")
print(df.isnull().sum()[df.isnull().sum() > 0])


Фінальна форма датасету: (1017209, 28)

Типи даних:
Store                                    int64
DayOfWeek                                int64
Date                            datetime64[ns]
Sales                                    int64
Customers                                int64
Open                                     int64
Promo                                    int64
StateHoliday                            object
SchoolHoliday                            int64
StoreType                               object
Assortment                              object
CompetitionDistance                    float64
CompetitionOpenSinceMonth                int64
CompetitionOpenSinceYear                 int64
Promo2                                   int64
Promo2SinceWeek                          int64
Promo2SinceYear                          int64
PromoInterval                           object
CompetitionDistance_missing              int64
CompetitionOpenSince_missing             int64
Year    

In [28]:
df.to_csv("rossmann_clean.csv", index=False)
print("Збережено: rossmann_clean.csv")
print(f"Розмір: {df.shape[0]:,} рядків, {df.shape[1]} колонок")


Збережено: rossmann_clean.csv
Розмір: 1,017,209 рядків, 28 колонок


**Примітка щодо `CompetitionOpenMonths`:** ця колонка містить NaN там, де `CompetitionOpenSince_missing == 1` (тобто дата відкриття конкурента невідома в вихідних даних). Це навмисно залишено як NaN, а не заповнено нулем — щоб на етапі моделювання/EDA можна було свідомо вирішити, як обробляти ці випадки (наприклад, виключити з кореляційного аналізу CompetitionDistance vs Sales, або імпутувати окремо). Прапорець `CompetitionOpenSince_missing` дозволяє завжди відфільтрувати ці рядки.

## Підсумок Етапу 1

- Дані об'єднано (`train` + `store`)
- Пропуски оброблено логічно (не наосліп), з прапорцями `_missing` для прозорості
- Додано похідні ознаки: `Year/Month/Day/WeekOfYear`, `CompetitionOpenMonths`, `Promo2Weeks`, `IsPromoMonth`, `SalesPerCustomer`
- Знайдено (але поки не видалено) потенційні викиди — рішення про їх обробку приймається на етапі EDA
- Результат збережено у `rossmann_clean.csv` для використання в наступних ноутбуках

**Наступний крок:** Етап 2 — EDA на рівні всієї мережі (тренди, сезонність, перші гіпотези).